In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

repo_root = Path.cwd().parent
sys.path.append(str(repo_root))

import src.utils.config as config
import src.utils.pdata_io as pdio

data_root, pdata_root, cc_data = pdio.load_project_context()

print(data_root)
print(pdata_root)
print("Number of animals:", len(cc_data))

/home/nmldata2/ccaw/Python
[LOADED] Project context: /mnt/pdata/Classical_Conditioning/_cache/project_context.pkl
/mnt/data/Classical_Conditioning
/mnt/pdata/Classical_Conditioning
Number of animals: 5


In [3]:
from pathlib import Path
import pandas as pd


def delete_npz_files_from_cc_data(
    cc_data,
    pdata_root,
    dry_run=True,
    recursive=True,
    phase_filter=None,
):
    """
    Find and optionally delete .npz files for animal/date folders listed in cc_data.

    Expected cc_data structure:
        cc_data[animal][date] = info_dict

    Expected processed-data structure:
        pdata_root / animal / date / ...
    
    Parameters
    ----------
    cc_data : dict
        Main data dictionary.

    pdata_root : str or Path
        Processed data root directory.

    dry_run : bool
        If True, only list files that would be deleted.
        If False, actually delete them.

    recursive : bool
        If True, search recursively under animal/date folder.
        If False, search only directly inside animal/date folder.

    phase_filter : str, list, tuple, set, or None
        If provided, only delete npz files for sessions matching this phase.
        Example: phase_filter="habituation"
    """

    pdata_root = Path(pdata_root)
    rows = []

    if phase_filter is not None:
        if isinstance(phase_filter, str):
            phase_filter = {phase_filter}
        else:
            phase_filter = set(phase_filter)

    for animal, days in cc_data.items():
        for date, info in days.items():

            phase = info.get("phase", "unknown")

            if phase_filter is not None and phase not in phase_filter:
                continue

            session_dir = pdata_root / animal / date

            if not session_dir.exists():
                rows.append({
                    "animal": animal,
                    "date": date,
                    "phase": phase,
                    "session_dir": str(session_dir),
                    "npz_file": None,
                    "status": "session_dir_missing",
                })
                continue

            if recursive:
                npz_files = list(session_dir.rglob("*.npz"))
            else:
                npz_files = list(session_dir.glob("*.npz"))

            if len(npz_files) == 0:
                rows.append({
                    "animal": animal,
                    "date": date,
                    "phase": phase,
                    "session_dir": str(session_dir),
                    "npz_file": None,
                    "status": "no_npz_found",
                })
                continue

            for f in npz_files:
                if dry_run:
                    status = "would_delete"
                else:
                    try:
                        f.unlink()
                        status = "deleted"
                    except Exception as e:
                        status = f"error: {e}"

                rows.append({
                    "animal": animal,
                    "date": date,
                    "phase": phase,
                    "session_dir": str(session_dir),
                    "npz_file": str(f),
                    "status": status,
                })

    df = pd.DataFrame(rows)

    n_files = df["npz_file"].notna().sum() if not df.empty else 0

    if dry_run:
        print(f"[DRY RUN] Found {n_files} .npz files that would be deleted.")
        print("Set dry_run=False to actually delete them.")
    else:
        print(f"[DELETE RUN] Processed {n_files} .npz files.")

    return df

In [7]:
npz_delete_report = delete_npz_files_from_cc_data(
    cc_data=cc_data,
    pdata_root=pdata_root,
    dry_run=False,
    recursive=True,
)

npz_delete_report

[DELETE RUN] Processed 163 .npz files.


,animal,date,phase,session_dir,npz_file,status
0,NML_04,2025_12_27,habituation,/mnt/pdata/Classical_Conditioning/NML_04/2025_...,/mnt/pdata/Classical_Conditioning/NML_04/2025_...,deleted
1,NML_04,2025_12_28,habituation,/mnt/pdata/Classical_Conditioning/NML_04/2025_...,/mnt/pdata/Classical_Conditioning/NML_04/2025_...,deleted
2,NML_04,2025_12_29,habituation,/mnt/pdata/Classical_Conditioning/NML_04/2025_...,/mnt/pdata/Classical_Conditioning/NML_04/2025_...,deleted
3,NML_04,2025_12_30,habituation,/mnt/pdata/Classical_Conditioning/NML_04/2025_...,/mnt/pdata/Classical_Conditioning/NML_04/2025_...,deleted
4,NML_04,2025_12_31,habituation,/mnt/pdata/Classical_Conditioning/NML_04/2025_...,/mnt/pdata/Classical_Conditioning/NML_04/2025_...,deleted
...,...,...,...,...,...,...
267,NML_08,2026_03_29,unknown,/mnt/pdata/Classical_Conditioning/NML_08/2026_...,None,session_dir_missing
268,NML_08,2026_03_30,unknown,/mnt/pdata/Classical_Conditioning/NML_08/2026_...,None,session_dir_missing
269,NML_08,2026_03_31,unknown,/mnt/pdata/Classical_Conditioning/NML_08/2026_...,None,session_dir_missing
270,NML_08,2026_04_01,unknown,/mnt/pdata/Classical_Conditioning/NML_08/2026_...,None,session_dir_missing
